# AgentCare — Flow Demo

Runs the real **Coordinator → Routing** handoff against the live Supabase database, one node at a time, so you can watch the shared state fill in and see what lands in the DB.

**Before running:** make sure this notebook uses the project's `.venv` kernel (top-right kernel picker), not a system Python — otherwise the imports won't resolve.

⚠️ These cells write real rows to the production database. The last cell cleans everything up, and the setup cell is idempotent (safe to re-run).

## 1. Setup

In [1]:
import os, sys
# make sure imports resolve from the repo root, not the notebooks/ folder
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from app.agents.coordinator import coordinator_node
from app.agents.routing import routing_node
from app.services.supabase.factory import get_supabase_client

client = get_supabase_client()

# We reuse the bootstrap admin's login id as a stand-in patient for the demo.
# (Semantically odd, but it's a real existing profiles.id, which is all the FK needs.)
TEST_USER_ID = '5b962a3b-5497-4b5a-b460-1e7359654a8b'
print('setup ok')

setup ok


In [3]:
def cleanup():
    """Remove any demo rows for TEST_USER_ID, children first (FK order). Safe to run anytime."""
    profiles = client.table('patient_profiles').select('id').eq('user_id', TEST_USER_ID).execute().data
    for p in profiles:
        runs = client.table('workflow_runs').select('id').eq('patient_id', p['id']).execute().data
        for run in runs:
            client.table('escalations').delete().eq('workflow_run_id', run['id']).execute()
        client.table('workflow_runs').delete().eq('patient_id', p['id']).execute()
        client.table('appointments').delete().eq('patient_id', p['id']).execute()
        client.table('patient_profiles').delete().eq('id', p['id']).execute()
    print('cleaned up', len(profiles), 'demo patient profile(s)')

# start from a clean slate so re-running the notebook doesn't hit the unique(user_id) constraint
cleanup()

cleaned up 0 demo patient profile(s)


## 2. Coordinator node

Resolves the patient identity, classifies intent with its own LLM call, creates the `workflow_run`, and decides who to hand off to (`delegated_to`).

In [4]:
# The patient's raw message — change this to try different flows.
RAW_REQUEST = 'I have a weird itchy rash on my arm and want to see someone about it.'

state = {
    'user_id': TEST_USER_ID,
    'patient_id': None,
    'workflow_run_id': None,
    'raw_request': RAW_REQUEST,
    'department': None,
    'slot_id': None,
    'appointment_id': None,
    'escalation_reason': None,
    'status': 'in_progress',
    'delegated_to': None,
}

coordinator_out = coordinator_node(state)
state.update(coordinator_out)   # merge the node's output back into shared state

print('coordinator returned:', coordinator_out)
print('delegated_to        :', state['delegated_to'])

coordinator returned: {'patient_id': '5b9c31bd-f408-4e48-83b2-f2da266f8cc3', 'workflow_run_id': 'f335a70b-0cb5-4d5b-94aa-b50cb9e91ff0', 'delegated_to': 'routing'}
delegated_to        : routing


In [5]:
# What actually landed in the workflow_runs table
wf = client.table('workflow_runs').select('*').eq('id', state['workflow_run_id']).single().execute().data
wf

{'id': 'f335a70b-0cb5-4d5b-94aa-b50cb9e91ff0',
 'patient_id': '5b9c31bd-f408-4e48-83b2-f2da266f8cc3',
 'current_step': 'coordinator',
 'state': {'summary': 'The patient wants to book an appointment to see someone about an itchy rash on their arm.',
  'intent_type': 'booking'},
 'status': 'in_progress',
 'created_at': '2026-07-21T15:00:21.957275+00:00',
 'updated_at': '2026-07-21T15:00:21.957275+00:00'}

## 3. Routing node

Only meaningful if the Coordinator delegated to `routing`. Fetches the live department list, then the LLM itself chooses between calling `RoutingDecision` (confident) or `escalate_request` (uncertain / possible emergency).

In [6]:
if state['delegated_to'] == 'routing':
    routing_out = routing_node(state)
    state.update(routing_out)
    print('routing returned:', routing_out)
else:
    print(f"Coordinator delegated to '{state['delegated_to']}', not routing — skipping.")

routing returned: {'department': 'Dermatology', 'delegated_to': 'appointment'}


In [7]:
# Final shared state after the two nodes have run
state

{'user_id': '5b962a3b-5497-4b5a-b460-1e7359654a8b',
 'patient_id': '5b9c31bd-f408-4e48-83b2-f2da266f8cc3',
 'workflow_run_id': 'f335a70b-0cb5-4d5b-94aa-b50cb9e91ff0',
 'raw_request': 'I have a weird itchy rash on my arm and want to see someone about it.',
 'department': 'Dermatology',
 'slot_id': None,
 'appointment_id': None,
 'escalation_reason': None,
 'status': 'in_progress',
 'delegated_to': 'appointment'}

In [8]:
# The workflow_run row after routing updated it (note current_step + status)
client.table('workflow_runs').select('*').eq('id', state['workflow_run_id']).single().execute().data

{'id': 'f335a70b-0cb5-4d5b-94aa-b50cb9e91ff0',
 'patient_id': '5b9c31bd-f408-4e48-83b2-f2da266f8cc3',
 'current_step': 'routing',
 'state': {'summary': "The patient's request about a weird itchy rash on the arm best fits the focus of the Dermatology department, which specializes in skin care.",
  'department': 'Dermatology'},
 'status': 'in_progress',
 'created_at': '2026-07-21T15:00:21.957275+00:00',
 'updated_at': '2026-07-21T15:00:21.957275+00:00'}

## 4. Try the escalation path

Re-run cells 2–3 with an emergency-sounding request to watch Routing choose `escalate_request` on its own instead of picking a department. For example, set:

```python
RAW_REQUEST = 'my chest hurts badly and I cant breathe properly'
```

Then run the Coordinator cell, then the Routing cell — you should see `status: 'escalated'`, `delegated_to: None`, and an `escalations` row created.

In [9]:
# See any escalations created for this workflow_run
client.table('escalations').select('*').eq('workflow_run_id', state['workflow_run_id']).execute().data

[]

## 5. Cleanup

Removes all demo rows created above.

In [10]:
cleanup()

cleaned up 1 demo patient profile(s)


In [2]:
from app.agents.checkpointer import setup_checkpointer

In [ ]:
setup_checkpointer()

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"


KeyboardInterrupt: 

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"

error connecting in 'pool-1': invalid percent-encoded token: "potterHarry%"
